In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

In [0]:
df.limit(5).display()

### Drop rows with missing customer ID

In [0]:
df = df.filter(col("cst_id").isNotNull())

In [0]:
df = df.drop_duplicates()

In [0]:
COL_RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}

for old_name, new_name in COL_RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

### Trimming what space in string values

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
df.select("gender").distinct().show()

In [0]:
df.select("marital_status").distinct().show()

In [0]:
df = (
    df
    .withColumn(
        "gender",
        F.when(F.upper(col("gender")) == "M", "Male")
        .when(F.upper(col("gender")) == "F", "Female")
        .otherwise("Unknown")
    )
    .withColumn(
        "marital_status",
        F.when(F.upper(col("marital_status")) == "S", "Single")
        .when(F.upper(col("marital_status")) == "M", "Married")
        .otherwise("Unknown")
    )
)
    

### Sanity check for transformed dataframe

In [0]:
df.limit(5).display()

### Writing to silver layer table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")

### Sanity checks for silver table

In [0]:
%sql
SELECT * FROM workspace.silver.crm_customers ORDER BY customer_id LIMIT 10